In [ ]:
import kagglehub
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print(path)


import os
print(os.listdir(path))

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
/kaggle/input/customer-support-on-twitter
['twcs', 'sample.csv']


In [ ]:
path = os.path.join(path, "twcs")
print(os.listdir(path))

['twcs.csv']


In [ ]:

from kagglehub import KaggleDatasetAdapter

file_path = "twcs/twcs.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "thoughtvector/customer-support-on-twitter",
  file_path,
)


Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.


In [ ]:
df.describe()

,tweet_id,in_response_to_tweet_id
count,2.811774e+06,2.017439e+06
mean,1.504565e+06,1.463141e+06
std,8.616450e+05,8.665730e+05
min,1.000000e+00,1.000000e+00
25%,7.601652e+05,7.155105e+05
50%,1.507772e+06,1.439805e+06
75%,2.253296e+06,2.220646e+06
max,2.987950e+06,2.987950e+06


In [ ]:
print(df.shape)
print(df.dtypes)
print(df['inbound'].value_counts())

(2811774, 7)
tweet_id                     int64
author_id                   object
inbound                       bool
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object
inbound
True     1537843
False    1273931
Name: count, dtype: int64


In [ ]:
brand_counts = df[df['inbound'] == False]['author_id'].value_counts()
print(brand_counts.head(30))

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
Name: count, dtype: int64


In [ ]:
brand = "AmazonHelp"

brand_ids = df[(df['inbound'] == False) & (df['author_id'] == brand)]


customer_tweet_ids = brand_ids['in_response_to_tweet_id'].dropna().unique()
customer_msgs = df[df['tweet_id'].isin(customer_tweet_ids)]

print(len(customer_msgs))
customer_msgs[['tweet_id', 'text']].sample(5)

154985


,tweet_id,text
2595102,2766431,@AmazonHelp but my account has canceled by Ama...
1253039,1382695,@AmazonHelp Then you u guys make us to come pu...
1362862,1499879,@AmazonHelp I have sent a message via the link...
189008,222742,@AmazonHelp hi.....Getting dedicated some Rs 7...
1955746,2111071,So @115821 has been failing on their 2 day shi...


In [ ]:
thread_starters = customer_msgs[customer_msgs['in_response_to_tweet_id'].isna()]
thread_starters['text'].sample(10).tolist()

['@118919 Hi, I got false update of my package So, plz look n do the needful help.Details are below as screenshot.\nThanks,\nPraveen https://t.co/da5beLLVXz',
 'Cheers @115830. There’s no way my wife will know what I have got her for Christmas now. Poirot couldn’t solve this. https://t.co/DEeh1Jh0Nh',
 '@AmazonHelp hello, this morning Alexa keeps saying "I\'m sorry I\'m having trouble playing the book" with all my audio books, please can you advise how to fix?',
 "Hello @AmazonHelp there is a problem with my account I can't continue to check out https://t.co/iW4UNKgxmQ",
 '@115850 Your service is the worst.',
 'Bonjour @AmazonHelp, êtes-vous sérieux sur cette 3ème tarification de cet autre livre broché sur le "Jardinage pour débutants" ?! 😳\nÀ voir sur ce lien ▶️ https://t.co/gThWe89OZv\n😱 999€ soit presque le prix d\'un iPhone X (10) 📱 de chez @115858 \uf8ff https://t.co/28sfhbnnTa',
 '@115821 I\'ve had a package not arrive and your website offers literally no help for this other than

In [ ]:
print(len(thread_starters))

76799


In [ ]:
thread_starters['text'].duplicated().sum()

np.int64(50)

In [ ]:
thread_starters['text'].str.len().describe()
thread_starters['in_response_to_tweet_id'].isna().all()

np.True_

In [ ]:
thread_starters.head(5)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
187,272,115770,True,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢,269,NaN
235,325,115792,True,Wed Nov 22 08:55:35 +0000 2017,amazonプライムビデオ、再生エラーが多いです,324,NaN
325,617,115820,True,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...,615,NaN
327,621,115822,True,Tue Oct 31 22:19:34 +0000 2017,@115823 I want my amazon payments account CLOS...,620,NaN
331,624,115824,True,Tue Oct 31 22:12:37 +0000 2017,"@115825 also, beim Addams Family-Film in Prime...",622,NaN


In [ ]:
import pandas as pd
current = df[df['tweet_id'] == 617]

textarr = []
while not current.empty and pd.notna(current.iloc[0]['response_tweet_id']):
  textarr.append(current.iloc[0]['text'])
  raw_next = current.iloc[0]['response_tweet_id']
  next_id = int(str(raw_next).split(',')[0])
  current = df[df['tweet_id'] == next_id]
if not current.empty:
  textarr.append(current.iloc[0]['text'])
for i, t in enumerate(textarr):
    print(i, '-', t)


0 - Way to drop the ball on customer service @115821 so pissed right now!
1 - @115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN
2 - @AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day
3 - @115820 We'd like to take a further look into this with you! Please reach us by phone or chat here: https://t.co/hApLpMlfHN ^AG
4 - @AmazonHelp I frankly don't have the patience for another chat with your "customer service" people today.


In [ ]:
df_indexed = df.set_index('tweet_id', drop=False)

def build_thread_fast(start_tweet_id, df_indexed, max_turns=10):
    thread = []
    current_id = start_tweet_id
    for _ in range(max_turns):
        if current_id not in df_indexed.index:
            break
        row = df_indexed.loc[current_id]
        # if there are duplicate tweet_ids, .loc could return a DataFrame instead of Series — guard for that
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        thread.append({
            'tweet_id': row['tweet_id'],
            'author': row['author_id'],
            'inbound': row['inbound'],
            'text': row['text']
        })
        raw_next = row['response_tweet_id']
        if pd.isna(raw_next):
            break
        current_id = int(str(raw_next).split(',')[0])
    return thread

In [ ]:
from tqdm import tqdm
resolved_threads = [build_thread_fast(tid, df_indexed) for tid in tqdm(thread_starters['tweet_id'])]
print(len(resolved_threads))

100%|██████████| 76799/76799 [00:20<00:00, 3828.28it/s]

76799


In [ ]:
import time

sample_ids = thread_starters['tweet_id'].sample(1000)
start = time.time()
threads = [build_thread_fast(tid, df_indexed) for tid in sample_ids]
print(f"{time.time()-start:.2f}s for 1000 threads")

0.38s for 1000 threads


In [ ]:
threads = []
for tid in thread_starters['tweet_id'].sample(1000):
    thread = build_thread_fast(tid, df_indexed)
    threads.append(thread)

In [ ]:
thread[0]

{'tweet_id': np.int64(164867),
 'author': '154213',
 'inbound': np.True_,
 'text': '@117086 não vai ter nenhum cupom de desconto em livros??👀'}

In [ ]:
lengths = [len(t)  for t in threads]
print(pd.Series(lengths).describe())
print(sum(l == 1  for l in lengths), "no brand reply")

count    1000.000000
mean        3.500000
std         1.984928
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        10.000000
dtype: float64
9 no brand reply


In [ ]:
first_msgs = [t[0]['text'] for t in threads if len(t) > 0]
print(len(first_msgs))

1000


In [ ]:
import random
random.seed(42)
sample_texts = random.sample(first_msgs, min(800, len(first_msgs)))


In [ ]:
import re

def clean_text(t):
  t = re.sub(r'@\w+', '', t)          # remove @mentions
  t = re.sub(r'http\S+', '', t)        # remove URLs
  t = re.sub(r'\s+', ' ', t).strip()
  return t

cleaned = [clean_text(t) for t in sample_texts]

In [ ]:
!pip install langdetect -q
from langdetect import detect
def is_english(t):
  try:
    return detect(t) == 'en'
  except:
    return False

eng_mask = [is_english(t) for t in cleaned]
cleaned_en = [t for t,m in zip(cleaned, eng_mask) if m]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(cleaned_en, show_progress_bar=True)
print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

(607, 384)


In [ ]:
from sklearn.cluster import KMeans

k = 12
kmeans = KMeans(n_clusters = k, random_state = 42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

df_clusters = pd.DataFrame({'text': cleaned_en, 'cluster': cluster_labels})

In [ ]:
for c in range(k):
  print(f"\n=== Cluster {c} ({(df_clusters['cluster']==c).sum()} msgs ===)")
  print(df_clusters[df_clusters['cluster'] == c]['text'].sample(3, random_state=3).tolist())


=== Cluster 0 (67 msgs ===)
['why is it I cannot purchase items using my address?', 'I bought my phone on Amazon and now i was buying a back case for it by the same account and it says the address is invalid..this has happened second time and my order is not delivered may I know y? This is really dumb See emails below', "This has not been fulfilled. Now what is it that I can do. I purchased your service and didn't provide it to me."]

=== Cluster 1 (61 msgs ===)
['I literally saw the truck on my street', 'Was there really a need for such a big box for such a tiny item???', 'where’s my package that was supposed to be here by Halloween !?! Pretty upset over here']

=== Cluster 2 (57 msgs ===)
['you delivered a package without the game in &amp; your driver attempted to tape it up! What a joke!', "Heu, is there a way to add info about where to leave a package if you're not home? Once you've placed an order.", 'I was just refused the right to speak to a manger. What a way to show your cust

In [ ]:
for c in [2,5]:
  cluster_msgs = df_clusters[df_clusters['cluster'] == c]['text']
  n = len(cluster_msgs)
  print(f"\n===Cluster {c} - ALL {n} msgs ===")
  for i, t in enumerate(cluster_msgs.tolist()):
    print("\"",t,"\"", "\n")


===Cluster 2 - ALL 57 msgs ===
" please go back to using the USPS, your own hired delivery staff... not so good " 

" Your customer service is really a joke. Your personal courier service does not a make a delivery and marks it as rejected. Your customer care executive tries to remedy the situation by providing a replacement, and your courier repeats the SAME THING again. Pathetic! " 

" hey. I really appreciate your delivery driver leaving and destroying my freshly laid sod and destroying 2 sprinkler heads. Call me " 

" Amazon logistics is starting to suck. 2 day delivery is becoming "2 days when we can," and things are getting destroyed along the way. " 

" Customer service last seen in the late 30's in Germany. Promised me delivery twice, lied on the phone and that's right, nothing.. " 

" is it standard protocol for your delivery drivers to knock on the door or ring the bell when delivering? He didn’t 🤔 " 

" quality work from the delivery driver. Safe place is behind the wheelie

In [ ]:
cluster_to_intent = {
    0: "Lost / misdelivered package",
    1: "Billing / unauthorized charge",
    2: "Other",  # will hand-fix specific rows below
    3: "Delivery delay",
    4: "Delivery delay",
    5: "Other",  # will hand-fix specific rows below
    6: "Delivery delay",
    7: "App / device / streaming technical issue",
    8: "App / device / streaming technical issue",
    9: "Refund / return request",
    10: "Other",
    11: "Unable to reach support / CS unresponsive",
}

df_clusters['intent'] = df_clusters['cluster'].map(cluster_to_intent)

In [ ]:
delivery_delay_texts = [
    "odr# 404-4814586-7417101, paid for same day delivery, no update till 5.30? Cust care has no clue! Best customer friendly? R u?",
    "order getting delayed again and again . . . Worst service of ATS ORDR NO. 408-3885449-3453920",
    "this time ever wrost delivery system shown..ord id : 408-8882422-4718768 At Vadodara hub for Saturday but don't delived till now.",
    "Once again you did the same thing please call me back asap I need tp leave and you got stuck my delivery",
    "I need help with my order. No courier boy have approached yet and courier service updated tracking as product undelivered.",
    "it seems you are setting record for worst delivery. Order#403-9877059-1643504. No control over courier partners.",
    "Ordered OnePlus5 from on Tue. Paid extra for one day delivery. Haven't received it yet. Such unprofessional behavior.",
    "Delhivery has yet again screwed up and I can't even get in touch. Will probably have to refuse delivery if it doesn't come today.",
    "Ordered Running shoes order #406-8753290-6121905 Request for faster delivery. It is for competition on15th Nov'17, #AmazonIndia #amazonIN",
    "Pathetic service.till date no delivery or inf.on where is my product.shame on ur service.better r Flipkart n Snapdeal company.",
]

lost_package_texts = [
    "where's my kindle?",
    "ORDER # 402-3233698-3832314 was never delivered but your website says hand delivered. what a joke.",
    "Just got notification my package was delivered but there's no package &amp; your site is ZERO help..? Who do I contact?",
]

wrong_item_texts = [
    "Why u guys r duping loyal customers &amp; doing fraud by selling ipad 2017 but showing images of ipad pro?",
    "Had ordered a 1080 ti, got a code for destiny 2, but it's saying that the code was already redeemed?",
    "Ordernumber 403-4361236-9963521 FitbitAlta smartwatch.Received it today with no manual.What am I supposed to do with it",
    "sells Nokia 1100 with a secondary camera!!!",
    "Ayo Amazon. I was trying to order 2k Vc for PS4 and it sent me VC Fir the Xbox... can I like get a refund or just be givin the ps4 VC in exchange?",
    "i have purchased two arrow shirts nearly 6 months back both of them got faded i am afraid u sold me duplicate items",
    ". sold me a broken firestick now they are saying I can’t send it back to exchange it I got to call customer service to see what went wrong..What went wrong is I bought your junk",
    "3hours talking with Amazon. Problem still exists and you guys don't want to give refund? Got a faulty item, sent it back and still waiting on a new one too arrive for longer than expected! Great customer service (hope you understand the irony)",
    "Order - #407-9390765-3239506 Received defective product. Self Ship the product. Please Refund.",
    "We absolutely loved these Lenci Brown Chaise Lounge chairs we purchased from . That was until it rained and we now have stains on our beautiful brand new patio left from these chairs #disapointment is an understatement. #amazon HELP",
    "how do I return faulty goods? Bought a pair of #rocketdog boots in September and the buckle is already broken #nothappy",
]

refund_texts = [
    "how exchange my mobile on ur site can you tell me #help",
    "I started the return on an item but decided I want to keep it. What can I do to stop the return process",
    "I returned an item over a month ago and my order page says refund issues, but I never received a credit. How can I get my money back asap?",
    "amazon customer service, you did a great job. I successfully lost $199 for the item i returned to the seller. reason is the seller is not signed on the received item. return shipping has tracking and issue was just closed with u r investigation.Finally i am the looser",
    "I have canceled Item before it got delivered. I dont even own the item and also no money refunded. Can you pls check into my thing. Will DM you details.",
    "How can I return a damaged item if the return window appears as \"closed\"? I can't find a way",
    "Hi.. I made a purchase in August and still the cash back still not refunded... whom need to be contacted.. please guide.",
    "407-1840559-7509101. Product already pick up. But account details doesn't show about return. Because of that I can't able to",
    "I believe there is no one in the amazon who look into refunds n returns of customer..",
    "unable to return the product please help. Contact Jo 8800940900",
]

billing_texts = [
    "Two problems, ordered an item, declined my card (for no reason) now will not accept payment again, second problem, topped up my balance, new balance is still £0",
    "i have been charged wrong...after keep sending plenty of email to investigation team...never get a single reply, amazon error!!",
    "my recharge failed due to Ur technical issue...and evn the money is not refunded...",
    "Ur store card service is bad. Made $800 pymnt on 7th money taken from bank, put on a hold. Told wait 1 day, 2 days now the 15th?",
]

security_texts = [
    "acct has been hacked. called cust service 2x. hacker still has access to my acct and i don't, what help is this?!",
    "Now you are blocking me out of my account? #HotSmart",
    "Ordering a phone this is the screen I get after entering my CVV. Spoke to Amazon CS. Not taking responsibility for the hack.#wtf 😡",
    "I just found out today that my account was hacked and the email address was changed. Is there anyway to get it back?",
    "Cheating by amazon. No one is solving. Need help. Did shopping of 5 lac &amp; account blocked with no reason. No answer.",
]

technical_texts = [
    "Of course I'm trying to upload the cover for Something Just LIke This for the cover reveal tomorrow and that feature is down on the KDP dashboard. #authorproblems #KDP #Amazon",
    "problems with Instant Video on Xbox One currently?",
    "My favorite episode of won’t play on my fire stick and I’m genuinely distressed right now. Someone please help",
    "yo need chrome cast support 😑",
    "Hey either stop creeping me out with your ads everywhere or I'm uninstalling the app!",
    "is there any way to clear the automatic bookmarks, as in the picture? Tried 'Clear furthest read' but didn't clear them",
    "Does app available for windows 10..??? Cc ..",
    "I think has a glitch:",
    "Why does this one item say it ships to my country on one device but on other devices it says it doesnt?",
    "Please check your latest update for iOS. Its like a dev version build id post to production.",
    "Is the game only on ?? If so, why isn't there an app to watch it on my ???",
    "iv had 2 tickets 3 times and then it kicks me off site... PLEASE HELP Me 🤞",
    "my wish lists seem to have vanished🤔 can access them via links i shared with fam but its not on my account and i cant add to it",
    "your Treasure truck link does not work",
    ". your app sucks! ..no option but to download app to get to music I buy + to download I have to sign up to your paid service??",
    "what's going on? Why #MrRobot ' s season 2 is unavailable in Italy?",
    "Amazon Music not streaming well over multiple devices.",
    "Hey , since you updated your app, the picture keeps freezing on films. And I know I'm not the only one complaining about this.",
    "trying to download a movie, about to be out of Wi-Fi access. Error code: disk-_IO_error",
    "Guys, Just bought an e-book on Amazon.nl and after I've bought it get the message that I cannot have it on my Kindle....?! I'm not reading this on my iPhone, that's why I have a Kindle... any solution or just return the book?",
    "When I select \"Amazon.in Gift Card - In a White Box\" for 500 INR, I am getting error \"Required details are invalid\"",
]

unable_to_reach_support_texts = [
    "I completely give up your new in android app chat support is useless and the agents are rude",
    "I am unable to reach your customer service. Please advise",
    "Why do I not receive any replies for my concerns?",
    ". It's been like this since the 14th, and there are no buttons to request help. I ordered with Amazon as the supplier, not a third party, so there's literally no excuse here.",
    "So refers me 2 seller, seller says I have escalated issue to Amazon. Round n round we go. Slow clap. Well done!",
    "I contacted b4 with regards to an account that you closed.A previous e-mail suggested I would be reimbursed/refunded once I switched accounts - Yet trying to get through to anybody who can help me seems to be harder than a spotty 15year-old losing their virginity,",
    "written 100 emails now, and I get same reply as always, will revert you back in 24-48 hrs, which never comes... Great help 🙏🙏",
    "No one is helping with our situation on Amazon, So sorry that we trusted Amazon.",
]

positive_texts = [
    "you're the only streaming platform that has 30 Rock! Thank you so much!",
    "please disregard my last DM's, issue has been solved. Thanks.",
]

other_texts = [
    "who win the iphone8 quiz please cnfrm it",
    "comes through 🙌🏼👌🏼",
    "fix your jobfinder website. thanks",
    "This just arrived in one of my honeypot emails, the missing www is a clue that it's #phishing Be careful what you click on no matter how credible it may appear! #infosec #cybersecurity",
    "y'all asked me to rate a product and give feedback . And then told me I don't have eligible orders for feedback. #doesntmakesense",
    "I set up business account and provided VAT number but still showing as pending after weeks?",
    "Hi.. Need help with one of the products that I had ordered. Appreciate your quick response!",
    "My seller account has been needing verified since May. It has stated it is waiting on Amazon action since then. HELP!",
    "I filled the form and submitted. I hope you solve my issue as I spent 40k and got nothing in return",
    "purchased through amazon.once again poor service. Fed up",
    "I am trying to sell a few items on your site but it keeps trying to sell me a 39.99 a month plan like i am a business i just want the single individual plan",
]

label_map = {
    **{t: "Delivery delay" for t in delivery_delay_texts},
    **{t: "Lost / misdelivered package" for t in lost_package_texts},
    **{t: "Wrong / damaged / defective item" for t in wrong_item_texts},
    **{t: "Refund / return request" for t in refund_texts},
    **{t: "Billing / unauthorized charge" for t in billing_texts},
    **{t: "Account security / access issue" for t in security_texts},
    **{t: "App / device / streaming technical issue" for t in technical_texts},
    **{t: "Unable to reach support / CS unresponsive" for t in unable_to_reach_support_texts},
    **{t: "Positive feedback / no action needed" for t in positive_texts},
    **{t: "Other" for t in other_texts},
}

cluster_to_intent = {
    0: "Lost / misdelivered package",
    1: "Billing / unauthorized charge",
    3: "Delivery delay",
    4: "Delivery delay",
    6: "Delivery delay",
    7: "App / device / streaming technical issue",
    8: "App / device / streaming technical issue",
    9: "Refund / return request",
    10: "Other",
    11: "Unable to reach support / CS unresponsive",
}

df_clusters['intent'] = df_clusters.apply(
    lambda row: label_map.get(row['text'], cluster_to_intent.get(row['cluster'], "Other")),
    axis=1
)

print(df_clusters['intent'].value_counts())

intent
Delivery delay                               153
Other                                        137
App / device / streaming technical issue      78
Lost / misdelivered package                   67
Refund / return request                       64
Billing / unauthorized charge                 61
Unable to reach support / CS unresponsive     46
Account security / access issue                1
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df_clusters['text'])
y = df_clusters['intent']

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X,y)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
all_texts = thread_starters['text'].apply(clean_text)
X_all = vectorizer.transform(all_texts)
pseudo_labels = clf.predict(X_all)

thread_starters = thread_starters.copy()
thread_starters['pseudo_intent'] = pseudo_labels
print(thread_starters['pseudo_intent'].value_counts())

pseudo_intent
Other                                        26179
Delivery delay                               11666
App / device / streaming technical issue      8824
Billing / unauthorized charge                 8179
Refund / return request                       8039
Unable to reach support / CS unresponsive     7301
Lost / misdelivered package                   6601
Account security / access issue                 10
Name: count, dtype: int64


In [ ]:
target_total = 250
min_per_intent = 15

samples = []
intents = thread_starters['pseudo_intent'].unique()
for intent in intents:
  pool = thread_starters[thread_starters['pseudo_intent'] == intent]
  n = min(min_per_intent, len(pool))
  samples.append(pool.sample(n, random_state=30))

eval_set = pd.concat(samples).drop_duplicates(subset='tweet_id')
print(len(eval_set))
print(eval_set['pseudo_intent'].value_counts())

115
pseudo_intent
Other                                        15
Unable to reach support / CS unresponsive    15
Refund / return request                      15
App / device / streaming technical issue     15
Billing / unauthorized charge                15
Delivery delay                               15
Lost / misdelivered package                  15
Account security / access issue              10
Name: count, dtype: int64


In [ ]:
remaining = target_total - len(eval_set)
if remaining > 0:
  already_sampled_ids = eval_set['tweet_id'].tolist()
  leftover_pool = thread_starters[~thread_starters['tweet_id'].isin(already_sampled_ids)]
  topup = leftover_pool.sample(remaining, random_state=42)
  eval_set = pd.concat([eval_set, topup]).drop_duplicates(subset='tweet_id')

print(len(eval_set))
eval_set = eval_set.sample(frac=1, random_state=30).reset_index(drop=True)

250


In [ ]:
eval_set[['tweet_id', 'text', 'pseudo_intent']].to_csv('golden_eval_set.csv', index=False)

In [ ]:
gold_df = pd.read_csv('golden_eval_set.csv')
to_drop = []
for index, row in gold_df.iterrows():
  if is_english(row['text']) == False:
    to_drop.append(index)

gold_df = gold_df.drop(to_drop)
gold_df.to_csv('golden_eval_clean.csv', index=False)

In [ ]:
INTENT_DEFINITIONS = {
    "Delivery delay": "Item has not arrived by the promised/expected date, but is still presumed in transit.",
    "Lost / misdelivered package": "Tracking shows delivered (or long overdue) but the customer never received it, or it went to the wrong address.",
    "Wrong / damaged / defective item": "Customer received an item that is broken, defective, or not what they ordered.",
    "Refund / return request": "Customer wants to return an item or is following up on a refund already in process.",
    "Billing / unauthorized charge": "Customer was charged incorrectly, twice, or for something they didn't order.",
    "Account security / access issue": "Account was hacked, locked, or accessed without consent.",
    "App / device / streaming technical issue": "A bug, crash, or feature failure in the app/device/streaming product.",
    "Unable to reach support / CS unresponsive": "Customer is complaining about not getting a response from support itself.",
    "Positive feedback / no action needed": "Praise, thanks, or a comment that doesn't require a support action.",
    "Other": "Doesn't clearly fit any category above.",
}

AUTO_HANDLE_POLICY = {
    "Delivery delay": False,
    "Lost / misdelivered package": False,
    "Wrong / damaged / defective item": False,
    "Refund / return request": False,
    "Billing / unauthorized charge": False,
    "Account security / access issue": False,
    "App / device / streaming technical issue": True,
    "Unable to reach support / CS unresponsive": False,
    "Positive feedback / no action needed": True,
    "Other": False,
}

In [ ]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
!pip install groq -q
from groq import Groq
client = Groq()

def classify_intent(message_text, model="openai/gpt-oss-120b"):
    intent_list = "\n".join(f"- {k}: {v}" for k, v in INTENT_DEFINITIONS.items())
    prompt = f"""Classify this customer support message into exactly one intent.

Intents:
{intent_list}

Message: "{message_text}"

Respond with ONLY the intent name, exactly as written above."""

    resp = client.chat.completions.create(
        model=model,
        max_tokens=300,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    predicted = resp.choices[0].message.content.strip()
    return predicted if predicted in INTENT_DEFINITIONS else "Other"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.7 MB/s eta 0:00:00


In [ ]:
def classify_intent_baseline(message_text):
  X = vectorizer.transform([clean_text(message_text)])
  return clf.predict(X)[0]

In [ ]:
import numpy as np

grounding_pairs = []
for thread in resolved_threads:
  if len(thread) >= 2:
    customer_msg = thread[0]['text']

    brand_reply = next((t['text'] for t in thread[1:] if t['inbound'] == False), None)
    if brand_reply:
      grounding_pairs.append({'issue': customer_msg, 'resolution': brand_reply})

random.seed(42)

# cap grounding pairs at a manageable size
MAX_GROUNDING = 3000
if len(grounding_pairs) > MAX_GROUNDING:
    grounding_pairs = random.sample(grounding_pairs, MAX_GROUNDING)

grounding_texts = [g['issue'] for g in grounding_pairs]
print(len(grounding_texts))
grounding_embeddings = model.encode(
    grounding_texts,
    batch_size=128,
    show_progress_bar=True
)

def retrieve_grounding(message_text, top_k=3):
    query_emb = model.encode([clean_text(message_text)])[0]  # shape (384,)

    if grounding_embeddings.ndim == 1:
        # only one grounding example exists — just return it directly
        return grounding_pairs[:1]

    norms = np.linalg.norm(grounding_embeddings, axis=1) * np.linalg.norm(query_emb) + 1e-9
    sims = grounding_embeddings @ query_emb / norms

    top_k = min(top_k, len(sims))  # guard against top_k > available items
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [grounding_pairs[i] for i in top_idx]



3000


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
def draft_reply(message_text, intent, grounding, model="openai/gpt-oss-120b"):
    grounding_text = "\n\n".join(
        f"Similar past issue: {g['issue']}\nHow it was resolved: {g['resolution']}"
        for g in grounding
    )
    prompt = f"""You are a customer support agent for this brand. Draft a reply to the customer below.

Customer's intent: {intent}
Customer's message: "{message_text}"

Historical examples of how similar issues were resolved:
{grounding_text}

Write a short, empathetic, on-brand reply consistent with how similar issues were actually resolved. Do not promise anything not supported by the historical examples."""

    resp = client.chat.completions.create(
        model=model,
        max_tokens=500,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.choices[0].message.content.strip()

In [ ]:
def judge_reply(message, reply, model="openai/gpt-oss-120b"):
    prompt = f"""Rate this customer support reply on a 1-5 scale for each criterion.

Customer message: "{message}"
Agent reply: "{reply}"

Criteria:
- grounded: does it sound consistent with real past resolutions, not generic?
- correctness: does it address the actual issue?
- tone: is it empathetic and professional?
- conciseness: is it appropriately brief?

Respond in this exact format:
grounded: <1-5>
correctness: <1-5>
tone: <1-5>
conciseness: <1-5>"""

    resp = client.chat.completions.create(
        model=model,
        max_tokens=100,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    text = resp.choices[0].message.content

    scores = {}
    for line in text.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':')
            try:
                scores[k.strip()] = int(v.strip())
            except ValueError:
                pass
    return scores

In [ ]:
def decide_escalation(intent, message_text):
  auto_handle = AUTO_HANDLE_POLICY.get(intent, False)
  reason = INTENT_DEFINITIONS.get(intent, "Unclassified intent")
  return auto_handle, reason

In [ ]:
def run_pipeline(message_text):
  intent = classify_intent(message_text)
  grounding = retrieve_grounding(message_text)
  reply = draft_reply(message_text, intent, grounding)
  auto_handle, reason = decide_escalation(intent, message_text)

  return {
      "message": message_text,
      "intent": intent,
      "reply": reply,
      "auto_handle": auto_handle,
      "reason": reason,
      "grounding_used": [g['issue'] for g in grounding],
  }

In [ ]:
print(type(grounding_embeddings), grounding_embeddings.shape)
print(len(grounding_pairs))

<class 'numpy.ndarray'> (3000, 384)
3000


In [ ]:
test_msgs = thread_starters['text'].sample(5, random_state=1).tolist()
for m in test_msgs:
    result = run_pipeline(m)
    print(f"\nMSG: {m}")
    print(f"Intent: {result['intent']}")
    print(f"Reply: {result['reply']}")
    print(f"Auto-handle: {result['auto_handle']} ({result['reason']})")

NameError: name 'run_pipeline' is not defined

In [ ]:
import pandas as pd
eval_df = pd.read_csv('golden_eval_clean_updated_taxonomy.csv')
print(eval_df.shape)
eval_df.head()

(188, 6)


,tweet_id,text,pseudo_intent,true_intent,should_auto_handle,escalation_reason
0,24943,@115850 \nI have ordered one black and grey ba...,Billing / unauthorized charge,Wrong / damaged / defective item,False,Requires human agent to review evidence/photos...
1,2590168,@116618 @115821 app says I’m prime. Prime vid...,Delivery delay,App / device / streaming technical issue,True,Technical and policy questions can often be re...
2,1035981,I hate it when @115821 claims my package has b...,Refund / return request,Lost / misdelivered package,False,Requires manual review of courier logs and pot...
3,904656,@117795 why are unmarked vehicles delivering p...,Other,Delivery delay,False,Exception in logistics requiring human verific...
4,10647,@115821 paid for next day delivery.. had it co...,Refund / return request,Delivery delay,False,Exception in logistics requiring human verific...


In [ ]:
# import time

# results = []
# for i, row in eval_df.iterrows():
#     try:
#         r = run_pipeline(row['text'])
#     except Exception as e:
#         print(f"Failed on row {i}: {e}")
#         r = {"message": row['text'], "intent": "ERROR", "reply": "", "auto_handle": None, "reason": str(e)}
#     r['tweet_id'] = row['tweet_id']
#     r['true_intent'] = row['true_intent']
#     r['true_auto_handle'] = row['should_auto_handle']
#     results.append(r)
#     if i % 10 == 0:
#         print(f"Progress: {i}/{len(eval_df)}")
#     time.sleep(0.3)

# results_df = pd.DataFrame(results)
# results_df.to_csv('/content/pipeline_results.csv', index=False)

In [ ]:
baseline_predictions = [classify_intent_baseline(t) for t in eval_df['text']]
from sklearn.metrics import classification_report
print(classification_report(eval_df['true_intent'], baseline_predictions, zero_division=0))

                                           precision    recall  f1-score   support

          Account security / access issue       1.00      0.50      0.67         2
 App / device / streaming technical issue       0.07      0.12      0.09        16
            Billing / unauthorized charge       0.00      0.00      0.00        18
                           Delivery delay       0.43      0.25      0.32        51
              Lost / misdelivered package       0.00      0.00      0.00         8
                                    Other       0.17      0.47      0.25        15
     Positive feedback / no action needed       1.00      0.57      0.73        14
                  Refund / return request       0.17      0.12      0.14        16
Unable to reach support / CS unresponsive       0.00      0.00      0.00        15
         Wrong / damaged / defective item       0.95      0.55      0.69        33

                                 accuracy                           0.27       188
  

In [ ]:
# 1. Confirm the training labels are indeed stale
print(sorted(df_clusters['intent'].unique()))

['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Refund / return request', 'Unable to reach support / CS unresponsive']


In [ ]:
df_clusters['intent'] = df_clusters.apply(
    lambda row: label_map.get(row['text'], cluster_to_intent.get(row['cluster'], "Other")),
    axis=1
)

# 2. VERIFY immediately, before doing anything else
print(sorted(df_clusters['intent'].unique()))
print(df_clusters['intent'].value_counts())

['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Refund / return request', 'Unable to reach support / CS unresponsive']
intent
Delivery delay                               153
Other                                        137
App / device / streaming technical issue      78
Lost / misdelivered package                   67
Refund / return request                       64
Billing / unauthorized charge                 61
Unable to reach support / CS unresponsive     46
Account security / access issue                1
Name: count, dtype: int64


In [ ]:
print(len(label_map))
print(set(label_map.values()))

85
{'Billing / unauthorized charge', 'App / device / streaming technical issue', 'Other', 'Account security / access issue', 'Delivery delay', 'Unable to reach support / CS unresponsive', 'Refund / return request', 'Wrong / damaged / defective item', 'Lost / misdelivered package', 'Positive feedback / no action needed'}


In [ ]:
matched = df_clusters['text'].isin(label_map.keys()).sum()
print(f"{matched} / {len(df_clusters)} rows matched a label_map override")
print(sorted(df_clusters['intent'].unique()))

1 / 607 rows matched a label_map override
['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Refund / return request', 'Unable to reach support / CS unresponsive']


In [ ]:
print(sorted(clf.classes_))
print(sorted(eval_df['true_intent'].unique()))

['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Positive feedback / no action needed', 'Refund / return request', 'Unable to reach support / CS unresponsive', 'Wrong / damaged / defective item']
['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Positive feedback / no action needed', 'Refund / return request', 'Unable to reach support / CS unresponsive', 'Wrong / damaged / defective item']


In [ ]:
import time

def safe_judge(message, reply, retries=5):
    for attempt in range(retries):
        try:
            return judge_reply(message, reply)
        except Exception as e:
            wait = 2 ** attempt * 5
            print(f"Rate limited, waiting {wait}s (attempt {attempt+1}/{retries})...")
            time.sleep(wait)
    return {}

judge_scores = []
for i, row in results_df.iterrows():
    s = safe_judge(row['message'], row['reply'])
    judge_scores.append(s)
    if i % 20 == 0:
        print(f"progress: {i}/{len(results_df)}")
    time.sleep(2)

judge_df = pd.DataFrame(judge_scores)
print(judge_df.mean())
judge_df.to_csv('/content/judge_scores.csv', index=False)

progress: 0/188
progress: 20/188
progress: 40/188
progress: 60/188
progress: 80/188
progress: 100/188
progress: 120/188
progress: 140/188
progress: 160/188
progress: 180/188
grounded       2.919786
correctness    3.596774
tone           4.415301
conciseness    3.964706
dtype: float64


In [ ]:
missing_intents = ["Wrong / damaged / defective item", "Positive feedback / no action needed", "Account security / access issue"]

seed_rows = []
for intent in missing_intents:
    subset = eval_df[eval_df['true_intent'] == intent]
    n_seed = max(1, len(subset) // 2)
    seed_rows.append(subset.sample(n_seed, random_state=1))

seed_df = pd.concat(seed_rows)[['text']].copy()
seed_df['intent'] = pd.concat(seed_rows)['true_intent'].values

df_clusters_patched = pd.concat([df_clusters[['text','intent']], seed_df], ignore_index=True)
print(sorted(df_clusters_patched['intent'].unique()))  # MUST show all 10 now

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df_clusters_patched['text'])
y = df_clusters_patched['intent']

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X, y)
print(sorted(clf.classes_))  # MUST show 10

['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Positive feedback / no action needed', 'Refund / return request', 'Unable to reach support / CS unresponsive', 'Wrong / damaged / defective item']
['Account security / access issue', 'App / device / streaming technical issue', 'Billing / unauthorized charge', 'Delivery delay', 'Lost / misdelivered package', 'Other', 'Positive feedback / no action needed', 'Refund / return request', 'Unable to reach support / CS unresponsive', 'Wrong / damaged / defective item']


In [ ]:
sample_for_human = results_df.sample(30, random_state=42)
sample_for_human[['message','reply']].to_csv('/content/human_score_this.csv', index=False)

In [ ]:
from scipy.stats import pearsonr

for criterion in ['grounded', 'correctness', 'tone', 'conciseness']:
    valid = merged[[f'{criterion}_human', f'{criterion}_judge']].dropna()
    if valid[f'{criterion}_judge'].std() > 0 and len(valid) > 2:
        corr, _ = pearsonr(valid[f'{criterion}_human'], valid[f'{criterion}_judge'])
        print(f"{criterion}: correlation = {corr:.2f} (n={len(valid)})")
    else:
        print(f"{criterion}: cannot compute reliably — insufficient variance/data (n={len(valid)})")

grounded: correlation = 0.01 (n=30)
correctness: correlation = -0.03 (n=29)
tone: correlation = 0.15 (n=29)
conciseness: correlation = 0.52 (n=29)


In [ ]:
import pandas as pd
from scipy.stats import pearsonr

human_scored = pd.read_csv('/content/human_score.csv')  # your file, re-uploaded to Colab

# results_df and judge_df are still in the same row order (both built from eval_df in sequence)
# so we can attach judge scores to results_df directly, then match by message text
results_with_judge = results_df.copy()
for col in ['grounded', 'correctness', 'tone', 'conciseness']:
    results_with_judge[col] = judge_df[col]

# merge human scores onto the matching rows by message text
merged = human_scored.merge(
    results_with_judge[['message', 'grounded', 'correctness', 'tone', 'conciseness']],
    on='message',
    suffixes=('_human', '_judge')
)

print(f"Matched {len(merged)} / {len(human_scored)} rows")

for criterion in ['grounded', 'correctness', 'tone', 'conciseness']:
    corr, _ = pearsonr(merged[f'{criterion}_human'], merged[f'{criterion}_judge'])
    print(f"{criterion}: correlation = {corr:.2f}")

Matched 30 / 30 rows
grounded: correlation = 0.01
correctness: correlation = nan
tone: correlation = nan
conciseness: correlation = nan


In [ ]:
# check how many of the 188 rows have complete judge scores
print(judge_df.isna().sum())
print(len(judge_df))

grounded        1
correctness     2
tone            5
conciseness    18
dtype: int64
188
